# `inspect_codegen` vs `inspect_llvm` / `inspect_asm`

Raw LLVM IR and assembly are the wrong interface for a human **or** an LLM agent.
A SAXPY kernel's `inspect_llvm` is ~28 kB of NRT + CPython wrappers. The fact you
needed — *no FMA, AVX2 ymm, 4.33 cycles/iter, port 1 bound* — is a 10-line card.

This notebook measures that on a real compile.


In [1]:
from numba import njit
import numpy as np
from numba.misc.codegen_card import inspect_codegen, compare_codegen

@njit
def saxpy(a, x, y, out):
    for i in range(x.shape[0]):
        out[i] = a * x[i] + y[i]

@njit(fastmath=True)
def saxpy_fm(a, x, y, out):
    for i in range(x.shape[0]):
        out[i] = a * x[i] + y[i]

n = 4096
args = (2.0, np.ones(n), np.ones(n), np.empty(n))
saxpy(*args)
saxpy_fm(*[np.copy(a) if isinstance(a, np.ndarray) else a for a in args])
sig = saxpy.signatures[0]
print("compiled", sig)

compiled (float64, Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True))


## Token cost: dumps vs card

`chars/4` is the usual cheap estimate. The ratio is what matters, not the tokenizer.


In [1]:
ir = saxpy.inspect_llvm(sig)
asm = saxpy.inspect_asm(sig)
card = inspect_codegen(saxpy, sig)
print(f"inspect_llvm  {len(ir):6d} chars   ~{len(ir)//4:5d} tokens")
print(f"inspect_asm   {len(asm):6d} chars   ~{len(asm)//4:5d} tokens")
print(f"dumps total   {len(ir)+len(asm):6d} chars   ~{(len(ir)+len(asm))//4:5d} tokens")
print(f"codegen card  {card.tokens.card_chars:6d} chars   ~{card.tokens.card_tokens:5d} tokens")
print(f"savings       {100*card.tokens.savings_vs_dumps:.1f}%")
print()
print("--- what an agent actually needs ---")
print(card.brief())

inspect_llvm   27668 chars   ~ 6917 tokens
inspect_asm    12088 chars   ~ 3022 tokens
dumps total    39756 chars   ~ 9939 tokens
codegen card     616 chars   ~  154 tokens
savings       98.5%

--- what an agent actually needs ---
codegen card: saxpy(float64, Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True))
ISA: AVX2  no FMA  273 insns  section=.ltext
top: mov 51, movabs 28, call 19, cmp 14, vmovsd 11, jne 11
MCA (hot loop, static; ignores cache/DRAM):
  raptorlake        4.33 cyc/iter  IPC 3.24  15 insns  bound ADLPPort01 3.50
note: no FMA (IEEE default; try @njit(fastmath=True))
note: hot loop is vmul+vadd, not FMA
note: MCA is compute-only; bandwidth-bound kernels run slower than this
tokens: card 154 vs llvm 6934 + asm 3022 (98% smaller than dumps)


## What `inspect_asm` actually contains

Most of the dump is the CPython wrapper (`PyArg_UnpackTuple`, `NRT_adapt_ndarray_from_python`).
The hot loop is ~15 instructions in the middle. Agents that "just read the asm" burn tokens
on ABI glue and then miss `vmulpd`/`vaddpd` (no FMA).


In [1]:
print("first 25 lines of inspect_asm (the thing agents usually eat):")
print("\n".join(asm.splitlines()[:25]))
print("...")
print(f"... {len(asm.splitlines())} lines total, wrappers start around .Lfunc_end0")

first 25 lines of inspect_asm (the thing agents usually eat):
	.file	"<string>"
	.section	.ltext,"axl",@progbits
	.globl	_ZN13_3cdynamic_3e5saxpyB2v1B38c8tJTIeFIjxB2IKSgI4CrvQClQZ6FczSBAA_3dEd5ArrayIdLi1E1C7mutable7alignedE5ArrayIdLi1E1C7mutable7alignedE5ArrayIdLi1E1C7mutable7alignedE
	.p2align	4
	.type	_ZN13_3cdynamic_3e5saxpyB2v1B38c8tJTIeFIjxB2IKSgI4CrvQClQZ6FczSBAA_3dEd5ArrayIdLi1E1C7mutable7alignedE5ArrayIdLi1E1C7mutable7alignedE5ArrayIdLi1E1C7mutable7alignedE,@function
_ZN13_3cdynamic_3e5saxpyB2v1B38c8tJTIeFIjxB2IKSgI4CrvQClQZ6FczSBAA_3dEd5ArrayIdLi1E1C7mutable7alignedE5ArrayIdLi1E1C7mutable7alignedE5ArrayIdLi1E1C7mutable7alignedE:
	movq	16(%rsp), %rax
	testq	%rax, %rax
	jle	.LBB0_19
	movq	120(%rsp), %rcx
	movq	64(%rsp), %rdx
	movq	8(%rsp), %rsi
	cmpq	$4, %rax
	jb	.LBB0_2
	movq	%rcx, %r8
	subq	%rsi, %r8
	cmpq	$128, %r8
	setb	%r8b
	movq	%rcx, %r9
	subq	%rdx, %r9
	cmpq	$128, %r9
	setb	%r9b
	orb	%r8b, %r9b
	je	.LBB0_5
.LBB0_2:
...
... 369 lines total, wrappers start around .Lfunc_en

## Value: IEEE vs `fastmath=True`

Same source. One flag. The card's job is to say what changed without a 20 kB diff.


In [1]:
print(compare_codegen(saxpy, saxpy_fm))

codegen compare
codegen card: saxpy(float64, Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True))
ISA: AVX2  no FMA  273 insns  section=.ltext
top: mov 51, movabs 28, call 19, cmp 14, vmovsd 11, jne 11
MCA (hot loop, static; ignores cache/DRAM):
  raptorlake        4.33 cyc/iter  IPC 3.24  15 insns  bound ADLPPort01 3.50
note: no FMA (IEEE default; try @njit(fastmath=True))
note: hot loop is vmul+vadd, not FMA
note: MCA is compute-only; bandwidth-bound kernels run slower than this
tokens: card 154 vs llvm 6934 + asm 3022 (98% smaller than dumps)

codegen card: saxpy_fm(float64, Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True), Array(float64, 1, 'C', False, aligned=True))
ISA: AVX2  FMA  273 insns  section=.ltext
top: mov 51, movabs 28, vmovsd 20, call 19, cmp 14, jne 11
MCA (hot loop, static; ignores cache/DRAM):
  raptorlake        3.67 cyc/iter  IPC 3.81  15 insns

## MCA is not wall time — that is the insight

SAXPY is 3 streams × 8 bytes. At 1e6 elements that is 24 MB, past L3.
`llvm-mca` says ~4.3 cycles/iter. Wall time is several times that. An agent that
only has `inspect_asm` will "optimize" the arithmetic. The card says: compute is
fine, you are buying bandwidth.


In [1]:
def bench(fn, n=1_000_000, repeats=15):
    x, y, out = np.ones(n), np.ones(n), np.empty(n)
    fn(2.0, x, y, out)
    best = 1e9
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(2.0, x, y, out)
        best = min(best, time.perf_counter() - t0)
    return best

w_ieee = bench(saxpy)
w_fm = bench(saxpy_fm)
print(f"wall n=1e6  ieee {w_ieee*1e6:.1f} us   fastmath {w_fm*1e6:.1f} us")

# MCA compute-only estimate: cyc/iter * (n / 16 elems) / 3.4 GHz
c_ieee = next(m.cycles_per_iter for m in inspect_codegen(saxpy).mca if m.ok)
print(f"MCA ieee @ 3.4 GHz compute-only: {c_ieee * (1_000_000/16) / 3.4e9 * 1e6:.1f} us")
print(f"wall / mca = {w_ieee / (c_ieee * (1_000_000/16) / 3.4e9):.1f}x")
print("ratio >> 1 => DRAM bound. FMA will not move wall time much.")
print(f"fastmath wall delta: {(w_ieee-w_fm)/w_ieee*100:+.1f}%")

wall n=1e6  ieee 309.6 us   fastmath 343.5 us
MCA ieee @ 3.4 GHz compute-only: 79.7 us
wall / mca = 3.9x
ratio >> 1 => DRAM bound. FMA will not move wall time much.
fastmath wall delta: -11.0%


## Agent protocol

| Instead of | Do |
|---|---|
| `print(fn.inspect_llvm())` | `print(fn.inspect_codegen())` |
| `print(fn.inspect_asm())` | same |
| eyeball two asm dumps | `compare_codegen(a, b)` |
| `llc` the whole module to aarch64 | card extracts kernel first; full module dies on `llvm.x86.atomic.sub.cc` |
| `perf` first | read MCA + the DRAM note; `perf` only if wall ≫ MCA |

CLI: `python -m numba.misc.codegen_card` and `--compare`.

Optional extras: `capstone`, `llvm-mca` on `PATH`.
